In [ ]:
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os

print("Running SD CBOS Web Scraping Tool v.1.0")

now=datetime.datetime.now()
filename= 'SD CBOS SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

scriptfolder=os.path.dirname(os.path.abspath(__file__))
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

regdict={'SD CBOS 1': 'https://cbos.gov.sd/en/content/operating-banks-sudan',
         'SD CBOS 2': 'https://cbos.gov.sd/en/content/authorized-exchange-bureaus'}

os.chdir(scriptfolder)
#print('The current folder is: {}\nThe temp folder is: {}'.format(scriptfolder, tempfolder))
chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory" : tempfolder, 
		"plugins.always_open_pdf_externally": True}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}
try:
	os.mkdir(tempfolder)
except:
	prevfiles=os.listdir(tempfolder)
	os.chdir(tempfolder)
	for prf in prevfiles:
		os.remove(prf)
	print('The directory tempfolder already exists.')
os.chdir(tempfolder)##only if files are going to be downloaded here

processdate=now.strftime('%Y-%m-%d')

for reg in regdict:
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(3)
    soup=BeautifulSoup(driver.page_source,"html.parser")
    div=soup.find("div",{"class":"region region-content"})
    if "SD CBOS 1" in reg:
        tbody = div.find("tbody")
        trs=tbody.find_all("tr")
        for tr in trs[1:]:
            print(tr.text)
            tds=tr.find_all("td")
            sqldict['Name'].append(tds[0].text.strip())
            sqldict['Address_1'].append(tds[1].text.strip())
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append('SD')
            sqldict['Cntry'].append('SD')
            sqldict['RegCode'].append('CBOS')
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListCode'].append('1')
            phones = tds[2].text.split('\n')
            if len(phones)==1:
                sqldict['Phone'].append(phones[0].strip())
            else:
                sqldict['Phone'].append('{}, {}'.format(phones[0].strip(),phones[1].strip()))
            faxes = tds[3].text.split('\n')
            if len(faxes)==1:
                sqldict['Fax'].append(faxes[0].strip())
            else:
                sqldict['Fax'].append('{}, {}'.format(faxes[0].strip(),faxes[1].strip()))
            for key in sqldict.keys():
                if len(sqldict['Name'])>len(sqldict[key]):
                    sqldict[key].append('')
    
    elif "SD CBOS 2" in reg:
        div=soup.find("div",{"class":"main-content-area-wrapper"})
        print(div.text)
        ols=div.find("ol")
        lis=ols.find_all("li")
        print(lis)
        for li in lis:
            sqldict['Name'].append(li.text.strip())
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append('SD')
            sqldict['Cntry'].append('SD')
            sqldict['RegCode'].append('CBOS')
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListCode'].append('2')
            for key in sqldict.keys():
                if len(sqldict['Name'])>len(sqldict[key]):
                    sqldict[key].append('')


os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()

sleep(3)

driver.quit()
    
    